# Chapter 1 — Model a coupled grounded resonator

TARGET API · CONVERGING · not executable on the current runtime

> **TARGET API / CONVERGING — not executable on the current runtime.**
> This Chapter authors a proposed circuit model and reviews its proposed
> diagram; it does not run a simulation.

This Chapter begins with the circuit diagram SCNSim will help you
produce, then teaches how to declare and render it. Its outcome is one
fully assembled Plan for the coupled resonator, then an ASCDLS diagram
and audit of that same Plan. Its three short Lessons declare the LC
resonator, assemble the root coupler and port, and review diagram
correspondence separately.

SCNSim turns an explicitly authored `CircuitPlan` into two independent
uses of the same circuit: ASCDLS can render its declared structure, and
a later `CircuitRun` can analyze it. A Plan stores components, values,
connections, subsystems, and Ports; it does not calculate anything, and
analysis does not require a prior diagram review.

## Lesson 1.1 — Grounded LC subsystem

### The circuit and this lesson’s question

We want a 110 fF / 5.8 nH grounded parallel resonator, coupled through 6
fF to a 50 ohm terminated measurement port. A `CircuitPlan` is the
explicit definition of that whole circuit, used independently for
drawing and later analysis. This author chooses to group the capacitor
and inductor into a resonator subsystem so it is one understandable part
of the complete circuit. Lesson 1.2 completes the circuit, and Lesson
1.3 reviews its authored diagram.

*Expected/design-intent sketch only—not current ASCDLS output or
certified evidence; any eventual real figure must derive from this same
complete Plan.*

``` text
50 ohm terminated Port ── signal boundary ── 6 fF ── resonator terminal
                                                   ├── 110 fF ── ground
                                                   └── 5.8 nH ── ground
```

The Plan will contain the port, coupler, and resonator together.
Grouping this LC now is an authoring choice for readability; a later
Chapter shows how the same kind of structure can be extracted into a
reusable Library component.

### Start the whole circuit and its resonator boundary

In [ ]:
from scnsim import CircuitPlan, components, units as u

plan = CircuitPlan(id="primitive_resonator")
resonator = plan.subsystem(id="resonator")

`plan` and `resonator` are Python variable names used in this notebook.
`id="primitive_resonator"` and `id="resonator"` are persistent authored
identities stored in the declaration. The Plan will contain the coupling
and measurement boundary; the grouped resonator contains its capacitor,
inductor, local wiring, and one exposed terminal.

### Add the fixed physical elements

These are fixed design values for this first model. `add()` registers a
native part in the resonator; it does **not** electrically connect that
part yet. `units as u` names the unit namespace, so `110.0 * u.fF` means
110 femtofarads and `5.8 * u.nH` means 5.8 nanohenries.

In [ ]:
capacitor = resonator.add(
    components.capacitor(id="capacitor", capacitance=110.0 * u.fF)
)
inductor = resonator.add(
    components.inductor(id="inductor", inductance=5.8 * u.nH)
)

The two returned elements are present but not wired. The next step says
that they share one resonator terminal and the resonator’s declared
ground.

`series`, `parallel`, and `branch` are component-containing assembly
declarations: each establishes real terminal incidence and a readable
authored structure. They are not labels applied after arbitrary wiring.
A `branch` can have the same electrical incidence as a series relation
from its start bus, but also declares the side-branch role; neither
spelling supplies current direction.

### Connect the LC branches in parallel

A bus names one electrical net. Here `resonator_bus` is the shared upper
endpoint of both branches; `resonator.ground` is their shared lower
endpoint. The parallel relation is the electrical statement, not an
inference from a future drawing. Each branch is a one-item Python tuple:
the trailing comma in `(capacitor,)` and `(inductor,)` tells Python that
it is a tuple, not merely a parenthesized element.

In [ ]:
resonator_bus = resonator.bus(id="terminal")
parallel_lc = resonator.parallel(
    id="parallel_lc",
    start=resonator_bus,
    branches=((capacitor,), (inductor,)),
    end=resonator.ground,
)

`parallel_lc` now records the grounded parallel LC. `resonator.ground`
is this subsystem’s intrinsic local declaration of the Plan’s canonical
reference: the relation, not a later parent call, grounds both LC return
ends. It is not an isolated ground; its local wiring remains inside the
resonator until the subsystem deliberately publishes its boundary.

## Lesson 1.2 — Assemble the parent circuit

### Expose the parent-facing resonator terminal

An exposed pin is the same electrical boundary, not an added component.
It is the handle the parent may connect without reaching into the
child’s capacitor, inductor, or local bus.

In [ ]:
terminal = resonator.expose_pin(id="terminal", at=resonator_bus)

The root circuit can now use `terminal`; the native LC leaves remain
owned by the resonator.

### Add the root buses and coupling capacitor

The parent has a signal-side bus and a separate resonator-side bus. It
first registers the 6 fF coupler, then states where that coupler
belongs.

A parent bus is optional when an existing public pin can be used
directly. This example deliberately gives the resonator-side assembly
node a root bus so it has a named assembly point and can later serve
public analysis. A Port needs a root `BusRef` or `TapRef`, and a named
root `.node` needs a named root bus; ordinary child wiring can instead
end directly at a public `PinRef`.

In [ ]:
signal_bus = plan.bus(id="signal_boundary")
resonator_root_bus = plan.bus(id="resonator_node")
coupling_cap = plan.add(
    components.capacitor(id="coupling_cap", capacitance=6.0 * u.fF)
)

Registering `coupling_cap` still does not connect it. The following
relation places it between the two root buses, then makes the
zero-component parent to child connection explicit.

In [ ]:
coupling = plan.series(
    id="coupling",
    start=signal_bus,
    elements=(coupling_cap,),
    end=resonator_root_bus,
)
plan.link(
    id="resonator_terminal",
    endpoints=(resonator_root_bus, terminal),
)

`coupling` preserves the coupler’s electrical order. `plan.link()` joins
its endpoint tuple as one zero-component direct connection, not a wire
part that becomes a new element or a post-hoc label. The tuple carries
no physical direction or layout order.

After this link, the root `resonator_root_bus` and child `terminal`
resolve to one electrical node: no extra node or component appears.
`resonator_root_bus` is a Python variable whose scoped authored id is
`resonator_node`; it gives the parent assembly structure and later
public analysis access, while `terminal` remains the child’s owned
boundary.

### Promote the terminated measurement boundary

The Port owns the already-defined 50 ohm load to the canonical reference
and its local ground glyph at the signal boundary; it has no public
Ground Pin. Its role and reference impedance are an authored measurement
condition, not a graphical label. The LC’s local ground and the Port
load resolve to the same canonical reference, but they are not one
graphical bus.

In [ ]:
signal_port = plan.add_port(
    id="signal_in",
    at=signal_bus,
    role="terminated",
    reference_impedance=50.0 * u.ohm,
)

The complete Plan now contains the target circuit: one grounded LC
subsystem, one 6 fF root coupler, and one terminated port.

## Lesson 1.3 — Review the authored diagram

### Render the complete authored circuit

The Automatic Semantic Circuit Diagram Layout System (ASCDLS) renders
explicitly authored components, wiring, hierarchy, public terminal, and
port facts. Automatic geometry is software work; it cannot infer
unwritten engineering intent or select an analytical View.

In [ ]:
from scnsim import CircuitDiagramSpec, Theme

diagram = plan.render_schematic(
    CircuitDiagramSpec(
        representation="authoring",
        theme=Theme.AUTO,
        show_parameter_values=True,
        show_provenance=True,
    )
)

The next cells review the picture, its separate Plan-projection audit,
and the few boundary handles that were intentionally authored.

In [ ]:
diagram.show()

In [ ]:
diagram.audit.show()

In [ ]:
from IPython.display import display

display(
    {
        "resonator parent-facing terminal": terminal,
        "terminated signal port": signal_port,
        "parallel LC relation": parallel_lc,
        "root coupling relation": coupling,
    }
)

Read the diagram in three layers. **Audit A** independently reconstructs
the electrical connectivity from the drawing and compares it with the
Plan/model. **Audit B** independently reconstructs subsystem ownership,
assembly, and terminal semantics from the drawing and compares them with
authored structure. **Human review C** asks whether this is the intended
circuit. The visible labelled region boundary/header witnesses inline
ownership; color is only a presentation aid. This render call omits
`parameters`, so its shown values are Plan baselines, not a selected
parameter override, solve, or View. The [selected-point
Chapter](04_define_parameters.qmd#ch4-point) shows how a diagram can use
an explicit point. Provenance binds the Plan, connectivity, and semantic
declaration identities; it does not certify intended circuit C or an
analysis request.

[Course map](../../docs/index.qmd) · [Next](02_solve_direct.qmd)